# 02 — Model

Inputs from notebook 01 (corrected **paid-renewal** label), persisted in `data/churn.duckdb`: `pred_points`, `cohorts`, `cohorts_s`.

arc: commitment + logistic floor -> engagement -> xgboost -> scope to paid -> lifecycle -> lock -> significance / split checks -> **model pick on net value** -> full-pop -> calibrate -> cost decision. full rationale in DECISIONS.md.

## setup


In [1]:
import duckdb, pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
con = duckdb.connect('../data/churn.duckdb')
q = lambda sql: con.execute(sql).df()
con.execute("SET enable_progress_bar = false")   # match phase A: quiet duckdb progress widgets


## load phase-A tables
*built + validated in 01 (corrected label) — just confirm they're present*


In [2]:
print(q('''SELECT split, count(*) AS n, count(DISTINCT msno) AS users,
                  round(avg(is_churn::int),4) AS churn_rate
           FROM cohorts_s GROUP BY split ORDER BY min(cohort_month)'''))


   split       n   users  churn_rate
0  train  300000  256907      0.1192
1    val  100000   96706      0.1368
2   test  100000   96314      0.0595


#### tables present
- corrected-label sample loaded: train 11.9% / val 13.7% / test 5.9% churn -> matches the phase A lock
- test base rate ~half of train/val = the real downward drift, carried through on purpose

## commitment features + logistic baseline
- cheapest signal first (the renewal transaction itself) = the PR-AUC floor before any behaviour
- is_cancel held out of features (leakage line); is_free derived from amount


In [3]:
q('''
CREATE OR REPLACE TABLE feat_commitment AS
SELECT c.msno, c.expiry, c.split, c.is_churn,
       t.is_auto_renew, t.payment_plan_days, t.actual_amount_paid, t.plan_list_price,
       (t.actual_amount_paid = 0)::int AS is_free,
       (t.plan_list_price - t.actual_amount_paid) AS discount,
       t.payment_method_id
FROM cohorts_s c
JOIN transactions t
  ON t.msno = c.msno
 AND t.transaction_date = strftime(c.gov_txn,'%Y%m%d')::INT
 AND t.membership_expire_date = strftime(c.expiry,'%Y%m%d')::INT
QUALIFY row_number() OVER (PARTITION BY c.msno, c.expiry ORDER BY t.actual_amount_paid DESC) = 1
''')
print(q('''SELECT is_auto_renew, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM feat_commitment WHERE split='train' GROUP BY is_auto_renew ORDER BY is_auto_renew'''))
print(q('''SELECT is_free, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM feat_commitment WHERE split='train' GROUP BY is_free ORDER BY is_free'''))


   is_auto_renew       n  churn_rate
0              0   52865       0.343
1              1  247135       0.071
   is_free       n  churn_rate
0        0  282884       0.075
1        1   17116       0.845


In [4]:
df = q("SELECT * FROM feat_commitment ORDER BY msno, expiry")
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount']
tr, va = df[df.split=='train'], df[df.split=='val']
ytr, yva = tr['is_churn'].astype(int), va['is_churn'].astype(int)
scaler = StandardScaler().fit(tr[feat_cols])
Xtr, Xva = scaler.transform(tr[feat_cols]), scaler.transform(va[feat_cols])
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
p_va = clf.predict_proba(Xva)[:,1]
print(f"val PR-AUC: {average_precision_score(yva, p_va):.4f}   (no-skill floor = {yva.mean():.4f})")
print(pd.Series(clf.coef_[0], index=feat_cols).sort_values())


val PR-AUC: 0.8586   (no-skill floor = 0.1368)
payment_plan_days    -0.829735
is_auto_renew        -0.624685
discount              0.009590
actual_amount_paid    0.466048
plan_list_price       0.475880
is_free               1.070701
dtype: float64


In [5]:
# interrogate the baseline: is the signal just easy segments? what about look-safe customers?
mask = (va['is_auto_renew']==1) & (va['is_free']==0)
y_seg, p_seg = yva[mask], p_va[mask]
print(f"safe segment: n={mask.sum()}, base={y_seg.mean():.4f}, PR-AUC={average_precision_score(y_seg, p_seg):.4f}")
te = df[df.split=='test']; yte = te['is_churn'].astype(int)
p_te = clf.predict_proba(scaler.transform(te[feat_cols]))[:,1]
print(f"test PR-AUC={average_precision_score(yte, p_te):.4f}  (test base rate={yte.mean():.4f})")


safe segment: n=77645, base=0.0109, PR-AUC=0.0230
test PR-AUC=0.4292  (test base rate=0.0595)


#### commitment floor
- terms alone split the obvious cases: auto_renew=0 churns 34% vs =1 7%; is_free 85% vs paid 7.5%
- logistic floor val PR-AUC 0.859 (no-skill 0.137) -- but driven by the free-trial split (is_free coef +1.07)
- the look-safe slice (auto_renew=1, paid, base 1.1%): PR-AUC 0.023 -> terms can't find these, behaviour next

## engagement features
- commitment reads obvious churns; need behavioural signal for look-safe customers
- recency / activity / completion / trend over a 60d window <= expiry (point-in-time)


In [6]:
q('''
CREATE OR REPLACE TABLE feat_engagement AS
WITH pts AS (
    SELECT msno, expiry, strftime(expiry,'%Y%m%d')::INT AS e_int,
           strftime(expiry-30,'%Y%m%d')::INT AS e_30, strftime(expiry-60,'%Y%m%d')::INT AS e_60
    FROM cohorts_s),
j AS (
    SELECT p.msno, p.expiry, p.e_int, p.e_30, l.date, l.num_25, l.num_50, l.num_75, l.num_985,
           l.num_100, l.num_unq, l.total_secs
    FROM pts p JOIN user_logs l ON l.msno=p.msno AND l.date>p.e_60 AND l.date<=p.e_int),
agg AS (
    SELECT msno, expiry,
           date_diff('day', strptime(max(date)::VARCHAR,'%Y%m%d')::DATE, expiry) AS recency_days,
           count(DISTINCT CASE WHEN date>e_30 THEN date END) AS active_days_30,
           count(DISTINCT CASE WHEN date<=e_30 THEN date END) AS active_days_prior,
           sum(CASE WHEN date>e_30 THEN total_secs ELSE 0 END) AS secs_30,
           sum(CASE WHEN date>e_30 THEN num_unq ELSE 0 END) AS unq_30,
           sum(CASE WHEN date>e_30 THEN num_100 ELSE 0 END) AS completed_30,
           sum(CASE WHEN date>e_30 THEN num_25+num_50+num_75+num_985+num_100 ELSE 0 END) AS plays_30
    FROM j GROUP BY msno, expiry)
SELECT *, completed_30/nullif(plays_30,0) AS completion_ratio,
       active_days_30/nullif(active_days_prior,0) AS activity_trend
FROM agg
''')
print(q("SELECT count(*) AS n_rows, count(DISTINCT msno) AS users FROM feat_engagement"))
print(q('''
SELECT CASE WHEN recency_days<=3 THEN '0-3d' WHEN recency_days<=14 THEN '4-14d'
            WHEN recency_days<=30 THEN '15-30d' ELSE '31-60d' END AS recency_bucket,
       count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
FROM feat_engagement e JOIN cohorts_s c USING (msno, expiry)
WHERE c.split='train' GROUP BY recency_bucket ORDER BY churn_rate
'''))


   n_rows   users
0  393349  317638
  recency_bucket       n  churn_rate
0           0-3d  190240       0.096
1         31-60d    6420       0.140
2          4-14d   32655       0.166
3         15-30d   10612       0.173


In [7]:
q('''
CREATE OR REPLACE TABLE model_data AS
SELECT c.*,
       (e.msno IS NOT NULL)::int AS has_activity_60d,
       coalesce(e.recency_days,60) AS recency_days,
       coalesce(e.active_days_30,0) AS active_days_30,
       coalesce(e.secs_30,0) AS secs_30,
       coalesce(e.unq_30,0) AS unq_30,
       coalesce(e.completion_ratio,0) AS completion_ratio,
       coalesce(e.activity_trend,0) AS activity_trend
FROM feat_commitment c
LEFT JOIN feat_engagement e USING (msno, expiry)
''')
print(q("SELECT count(*) AS n, count(*) FILTER (WHERE has_activity_60d=0) AS silent FROM model_data"))
print(q('''SELECT has_activity_60d, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM model_data WHERE split='train' GROUP BY has_activity_60d ORDER BY has_activity_60d'''))


        n  silent
0  500000  106651
   has_activity_60d       n  churn_rate
0                 0   60073       0.157
1                 1  239927       0.110


In [8]:
df = q("SELECT * FROM model_data ORDER BY msno, expiry")
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
tr, va, te = df[df.split=='train'], df[df.split=='val'], df[df.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
scaler = StandardScaler().fit(tr[feat_cols])
Xtr, Xva, Xte = scaler.transform(tr[feat_cols]), scaler.transform(va[feat_cols]), scaler.transform(te[feat_cols])
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
p_va, p_te = clf.predict_proba(Xva)[:,1], clf.predict_proba(Xte)[:,1]
print(f"val  PR-AUC: {average_precision_score(yva,p_va):.4f}")
print(f"test PR-AUC: {average_precision_score(yte,p_te):.4f}")
m = (va.is_auto_renew==1) & (va.is_free==0)
print(f"SAFE segment val: n={m.sum()}, base={yva[m].mean():.4f}, PR-AUC={average_precision_score(yva[m],p_va[m]):.4f}")
print(pd.Series(clf.coef_[0], index=feat_cols).sort_values())


val  PR-AUC: 0.8969
test PR-AUC: 0.5178
SAFE segment val: n=77645, base=0.0109, PR-AUC=0.0329
payment_plan_days    -1.039390
is_auto_renew        -0.689585
active_days_30       -0.344670
unq_30               -0.070804
secs_30               0.017082
discount              0.044820
completion_ratio      0.139663
activity_trend        0.152655
has_activity_60d      0.387526
recency_days          0.439117
actual_amount_paid    0.565547
plan_list_price       0.593436
is_free               1.038551
dtype: float64


In [9]:
# collinearity: read coefs with care
eng   = ['has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
price = ['payment_plan_days','actual_amount_paid','plan_list_price','discount','is_free']
print(tr[eng].corr().round(2)['recency_days'].sort_values())
print()
print(tr[price].corr().round(2))


has_activity_60d   -0.95
completion_ratio   -0.83
active_days_30     -0.72
unq_30             -0.42
activity_trend     -0.23
secs_30             0.02
recency_days        1.00
Name: recency_days, dtype: float64

                    payment_plan_days  actual_amount_paid  plan_list_price  \
payment_plan_days                1.00                0.86             0.97   
actual_amount_paid               0.86                1.00             0.89   
plan_list_price                  0.97                0.89             1.00   
discount                         0.20               -0.26             0.22   
is_free                         -0.10               -0.29            -0.13   

                    discount  is_free  
payment_plan_days       0.20    -0.10  
actual_amount_paid     -0.26    -0.29  
plan_list_price         0.22    -0.13  
discount                1.00     0.34  
is_free                 0.34     1.00  


#### + engagement
- silent (no activity in 60d) churns 15.7% vs active 11.0%; recency leads (0-3d 9.6% rising to ~17% by 15-30d; non-monotonic at the sparse 31-60d bucket)
- logistic +engagement: val 0.897; safe slice 0.033 (up from 0.023) -> small but real lift where it's hard
- features are collinear (recency vs has_activity -0.95; pricing cluster up to 0.97) -> trust importance/direction, not coef magnitudes

## xgboost
- nonlinear + interactions, same 13 feats
- read gain importance with care (collinearity + the is_free root split inflate it)


In [10]:
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
df = q("SELECT * FROM model_data ORDER BY msno, expiry")
tr, va, te = df[df.split=='train'], df[df.split=='val'], df[df.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
clf = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain', tree_method='hist', n_jobs=1, random_state=42)
clf.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
p_va = clf.predict_proba(va[feat_cols])[:,1]; p_te = clf.predict_proba(te[feat_cols])[:,1]
print(f"trees stopped at: {clf.best_iteration}")
print(f"val  PR-AUC: {average_precision_score(yva,p_va):.4f}")
print(f"test PR-AUC: {average_precision_score(yte,p_te):.4f}")
m = (va.is_auto_renew==1) & (va.is_free==0)
print(f"SAFE segment val: PR-AUC={average_precision_score(yva[m],p_va[m]):.4f}")
print(pd.Series(clf.feature_importances_, index=feat_cols).sort_values(ascending=False))


trees stopped at: 77
val  PR-AUC: 0.9176
test PR-AUC: 0.5839
SAFE segment val: PR-AUC=0.0386
is_free               0.635039
actual_amount_paid    0.172291
is_auto_renew         0.090862
discount              0.050106
active_days_30        0.009107
plan_list_price       0.009088
recency_days          0.008834
activity_trend        0.006202
payment_plan_days     0.005781
unq_30                0.005730
has_activity_60d      0.004327
secs_30               0.001419
completion_ratio      0.001214
dtype: float32


In [11]:
# interrogation: is the topline just a free-trial detector? drop trials and re-score
for name, dfx, y, p in [('val', va, yva, p_va), ('test', te, yte, p_te)]:
    paid = (dfx.is_free == 0)
    print(f"{name} paid-only: n={paid.sum()}, base={y[paid].mean():.4f}, "
          f"PR-AUC={average_precision_score(y[paid], p[paid]):.4f}   (topline {name}: {average_precision_score(y, p):.4f})")


val paid-only: n=89089, base=0.0389, PR-AUC=0.4150   (topline val: 0.9176)
test paid-only: n=98267, base=0.0465, PR-AUC=0.4010   (topline test: 0.5839)


#### xgboost + the free-trial tell
- xgb all-pop: val 0.918 / test 0.586; gain dominated by is_free (0.65, the root split) -> topline is largely a trial detector
- drop trials, re-score paid-only: val 0.416 / test 0.400 -> the real paid task sits at ~0.40, not ~0.9

## scope to paid — DECISION
- model targets paid only (is_free=0); free trials (~93% churn) -> conversion RULE, not a retention offer
- why: cost model assumes a paying customer; trials are trivially separable + have different economics; scoping de-inflates the metric and focuses capacity
- full rationale in DECISIONS.md


In [12]:
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
dfp = q("SELECT * FROM model_data WHERE is_free = 0 ORDER BY msno, expiry")
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
safe = (va.is_auto_renew == 1)
sc = StandardScaler().fit(tr[feat_cols])
lr = LogisticRegression(max_iter=2000).fit(sc.transform(tr[feat_cols]), ytr)
lr_va, lr_te = lr.predict_proba(sc.transform(va[feat_cols]))[:,1], lr.predict_proba(sc.transform(te[feat_cols]))[:,1]
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain', tree_method='hist', n_jobs=1, random_state=42)
xgb.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
xg_va, xg_te = xgb.predict_proba(va[feat_cols])[:,1], xgb.predict_proba(te[feat_cols])[:,1]
print(f"paid base rate    : val {yva.mean():.4f}  test {yte.mean():.4f}")
print(f"logistic-paid     : val {average_precision_score(yva,lr_va):.4f}  test {average_precision_score(yte,lr_te):.4f}")
print(f"xgboost-paid      : val {average_precision_score(yva,xg_va):.4f}  test {average_precision_score(yte,xg_te):.4f}")
print(f"xgboost safe slice: val {average_precision_score(yva[safe],xg_va[safe]):.4f}  (n={safe.sum()})")
print(f"trees stopped at  : {xgb.best_iteration}")
print(pd.Series(xgb.feature_importances_, index=feat_cols).sort_values(ascending=False))


paid base rate    : val 0.0389  test 0.0465
logistic-paid     : val 0.3337  test 0.2840
xgboost-paid      : val 0.4179  test 0.4021
xgboost safe slice: val 0.0408  (n=77645)
trees stopped at  : 62
is_auto_renew         0.676477
discount              0.076889
plan_list_price       0.050255
actual_amount_paid    0.046554
payment_plan_days     0.037887
recency_days          0.034419
has_activity_60d      0.020457
activity_trend        0.019195
active_days_30        0.015744
unq_30                0.014833
secs_30               0.004050
completion_ratio      0.003239
dtype: float32


#### paid model (12-feat)
- paid base val 3.9% / test 4.7%; xgb-paid val 0.417 / test 0.400 vs logistic 0.334 / 0.284 -> trees add ~0.08-0.12 over linear
- is_auto_renew dominates gain (0.69); safe slice still ~0.04 -> terms+behaviour only modestly crack the look-safe segment

## lifecycle features
- tenure_days, n_prior_cycles as-of-expiry (point-in-time): does loyalty / history add over renewal + behaviour?


In [13]:
con.execute('''
CREATE OR REPLACE TABLE feat_lifecycle AS
WITH pts AS (SELECT msno, expiry FROM cohorts_s),
     tx  AS (SELECT msno, strptime(transaction_date::VARCHAR,'%Y%m%d')::DATE AS txn_date FROM transactions)
SELECT p.msno, p.expiry,
       date_diff('day', MIN(t.txn_date), p.expiry) AS tenure_days,
       COUNT(DISTINCT t.txn_date)                   AS n_prior_cycles
FROM pts p JOIN tx t ON t.msno = p.msno AND t.txn_date <= p.expiry
GROUP BY p.msno, p.expiry
''')
print(q("SELECT COUNT(*) AS n FROM feat_lifecycle"))
print(q("SELECT MIN(tenure_days) lo, MAX(tenure_days) hi, MIN(n_prior_cycles) clo, MAX(n_prior_cycles) chi FROM feat_lifecycle"))
print(q('''
SELECT q AS tenure_quartile, ROUND(AVG(is_churn::INT),3) AS churn_rate, COUNT(*) AS n FROM (
  SELECT c.is_churn, NTILE(4) OVER (ORDER BY f.tenure_days) AS q
  FROM feat_lifecycle f JOIN cohorts_s c USING (msno, expiry)
) GROUP BY q ORDER BY q
'''))


        n
0  499917
   lo   hi  clo  chi
0   0  789    1   48
   tenure_quartile  churn_rate       n
0                1       0.273  124980
1                2       0.053  124979
2                3       0.068  124979
3                4       0.049  124979


#### lifecycle signal
- newest-tenure quartile churns 27% vs ~5% for older quartiles -> strong first-cycle risk
- tenure 0-789d, n_prior_cycles 1-48 (as-of-expiry, point-in-time)

In [14]:
num_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
            'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend',
            'tenure_days','n_prior_cycles']
cat_cols = ['payment_method_id']; feat_cols = num_cols + cat_cols
dfp = q('''SELECT m.*, f.tenure_days, f.n_prior_cycles FROM model_data m
           LEFT JOIN feat_lifecycle f USING (msno, expiry) WHERE m.is_free = 0 ORDER BY msno, expiry''')
for c in cat_cols: dfp[c] = dfp[c].astype('category')
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
safe = (va.is_auto_renew == 1)
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain',
                    enable_categorical=True, tree_method='hist', n_jobs=1, random_state=42)
xgb.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
xg_va, xg_te = xgb.predict_proba(va[feat_cols])[:,1], xgb.predict_proba(te[feat_cols])[:,1]
print(f"paid base          : val {yva.mean():.4f}  test {yte.mean():.4f}")
print(f"xgboost +lifecycle : val {average_precision_score(yva,xg_va):.4f}  test {average_precision_score(yte,xg_te):.4f}")
print(f"safe slice (AR=1)  : val {average_precision_score(yva[safe],xg_va[safe]):.4f}")
print(f"trees stopped at   : {xgb.best_iteration}")
print(pd.Series(xgb.feature_importances_, index=feat_cols).sort_values(ascending=False))


paid base          : val 0.0389  test 0.0465
xgboost +lifecycle : val 0.4785  test 0.4450
safe slice (AR=1)  : val 0.3257
trees stopped at   : 223
payment_method_id     0.229737
n_prior_cycles        0.192717
is_auto_renew         0.170764
recency_days          0.076933
payment_plan_days     0.062228
tenure_days           0.056633
plan_list_price       0.046732
activity_trend        0.032332
actual_amount_paid    0.032227
active_days_30        0.031093
discount              0.029974
unq_30                0.020477
secs_30               0.008262
completion_ratio      0.007418
has_activity_60d      0.002472
dtype: float32


In [15]:
# 1) censoring fingerprint: is tenure bounded by the split (i.e. by calendar)?
print(dfp.groupby('split')['tenure_days'].agg(['mean','max']))

# 2) ablation: add ONE family at a time, watch best_iteration + val/safe
base12 = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
          'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
def run(cols, cats=()):
    for c in cats: dfp[c] = dfp[c].astype('category')
    m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='aucpr', early_stopping_rounds=50, enable_categorical=bool(cats),
                      tree_method='hist', n_jobs=1, random_state=42)
    m.fit(tr[cols], ytr, eval_set=[(va[cols], yva)], verbose=False)
    pv = m.predict_proba(va[cols])[:,1]
    return f"iters={m.best_iteration:>4}  val={average_precision_score(yva,pv):.4f}  safe={average_precision_score(yva[safe],pv[safe]):.4f}"

print("base12             :", run(base12))
print("base12 + lifecycle :", run(base12+['tenure_days','n_prior_cycles']))
print("base12 + paymethod :", run(base12+['payment_method_id'], cats=['payment_method_id']))


             mean  max
split                 
test   446.097011  789
train  235.364941  608
val    391.760607  699


base12             : iters=  62  val=0.4179  safe=0.0408


base12 + lifecycle : iters= 190  val=0.4130  safe=0.0883


base12 + paymethod : iters= 141  val=0.4582  safe=0.2812


#### lifecycle / payment_method_id -- DECISION (resolved on the corrected label, selection on val)
- payment_method_id POISONS (high-card categorical, memorised, temporal mix shift) -> DROP
- lifecycle splits two ways on val: topline -0.0073 [-0.0164,+0.0022] WITHIN noise (no real topline penalty), safe-slice +0.0437 [+0.0277,+0.0602] significantly BETTER; and the 14-feat overfits (train 0.66 / test 0.40 via the tenure calendar-clock; base12 gap ~0, test even >= train)
- decided on the money, not the AUC, on VAL (not test -- selection moved here after an earlier test-based run was found to be a procedure error): base12 nets NT$227,561 vs 14-feat NT$221,689 at val break-even 0.303, contacting 224 FEWER customers at higher precision (0.474 vs 0.457) and higher prec@budget on both cuts -> dominance, not just a net edge -> LOCK base12. test (below) is report-only confirmation, opened once.
- the safe-slice gain looked actionable, but the arithmetic closes it: the top bin (qcut collapses to 8, not 10 -- ties in the isotonic step) needs save_rate 2.156 to clear break-even, 0 of 77,645 clear the cut -> not a phase-D segment, not actionable at the current offer cost. binding constraint is the offer, not the model.
- discovery arc kept: hypothesised lifecycle helps -> topline-neutral but overfits via the tenure clock, sharpens a segment the current offer economics can't reach -> dropped from the scorer, closed (not parked)

In [16]:
# lifecycle on test -- report-only; selection locked on val OOF backtest
safe_te = (te.is_auto_renew == 1)
def run_full(cols):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=1, random_state=42)
    m.fit(tr[cols], ytr, eval_set=[(va[cols], yva)], verbose=False)
    pv, pt = m.predict_proba(va[cols])[:,1], m.predict_proba(te[cols])[:,1]
    return (f"iters={m.best_iteration:>4} | val={average_precision_score(yva,pv):.4f} "
            f"test={average_precision_score(yte,pt):.4f} | "
            f"safe_val={average_precision_score(yva[safe],pv[safe]):.4f} "
            f"safe_test={average_precision_score(yte[safe_te],pt[safe_te]):.4f}")

print("base12             :", run_full(base12))
print("base12 + lifecycle :", run_full(base12 + ['tenure_days','n_prior_cycles']))

base12             : iters=  62 | val=0.4179 test=0.4021 | safe_val=0.0408 safe_test=0.0377


base12 + lifecycle : iters= 190 | val=0.4130 test=0.3941 | safe_val=0.0883 safe_test=0.0796


## locked model
- paid population, **12 features** (base12); payment_method_id and lifecycle both dropped from the scorer (decision above)
- the 14-feat is still fit alongside, only to drive the paired + safe-slice bootstrap that made the call
- store base12 train/val/test preds -> calibrator + honest eval + cost run on these

In [17]:
base12 = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
          'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
feat_cols = base12                                      # LOCKED scorer = 12 feats (lifecycle dropped, see decision above)
feat14    = base12 + ['tenure_days','n_prior_cycles']  # 14-feat kept ONLY to test lifecycle's effect
dfp = q('''SELECT m.*, f.tenure_days, f.n_prior_cycles FROM model_data m
           LEFT JOIN feat_lifecycle f USING (msno, expiry) WHERE m.is_free = 0 ORDER BY msno, expiry''')
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)

def fit_xgb(cols):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=1, random_state=42)
    m.fit(tr[cols], ytr, eval_set=[(va[cols], yva)], verbose=False)
    return m

xgb     = fit_xgb(feat_cols)                            # the locked 12-feat scorer (we defend this)
xgb_14  = fit_xgb(feat14)                               # reference only -> paired lifecycle test
p_tr = xgb.predict_proba(tr[feat_cols])[:,1]            # train preds -> overfit gap
p_va = xgb.predict_proba(va[feat_cols])[:,1]            # val preds   -> calibrator
p_te = xgb.predict_proba(te[feat_cols])[:,1]            # test preds  -> honest eval + cost
p_te_14 = xgb_14.predict_proba(te[feat14])[:,1]         # 14-feat test -> lifecycle bootstrap only
print(f"locked base12 : test PR-AUC {average_precision_score(yte,p_te):.4f}  "
      f"(14-feat {average_precision_score(yte,p_te_14):.4f})  trees {xgb.best_iteration}")

locked base12 : test PR-AUC 0.4021  (14-feat 0.3941)  trees 62


In [18]:
# B1 -- val cost selection: base12 vs 14-feat, compared on VAL outcomes (not test).
# calibration can't be fit and scored on the same rows (that overstates precision/net) ->
# 2-fold OOF isotonic within val: fit on one half, calibrate the other, swap, concatenate.
# this is a calibration-integrity split INSIDE val, not a new train/val/test boundary --
# rule #5's point-in-time discipline is about the outer split, already satisfied.
from sklearn.model_selection import KFold

def oof_calibrate(raw, y, seed=42):
    cal = np.empty_like(raw)
    for fit_idx, hold_idx in KFold(n_splits=2, shuffle=True, random_state=seed).split(raw):
        iso = IsotonicRegression(out_of_bounds='clip').fit(raw[fit_idx], y[fit_idx])
        cal[hold_idx] = iso.predict(raw[hold_idx])
    return cal

p_va_14 = xgb_14.predict_proba(va[feat14])[:,1]                       # 14-feat val raw scores
cal_va, cal_va_14 = oof_calibrate(p_va, yva.values), oof_calibrate(p_va_14, yva.values)

offer, save, horizon = 150, 0.30, 12
pmv = va.payment_plan_days > 0
arpu_val  = (va.loc[pmv,'actual_amount_paid'] / va.loc[pmv,'payment_plan_days'] * 30).median()   # val-derived, NOT the deployed constant
value_val = arpu_val * horizon
be_val    = offer / (save * value_val)
yv_val    = yva.values

def decide_val(cal):
    mask = cal >= be_val
    n, tp = int(mask.sum()), int(yv_val[mask].sum())
    row = {'contacted': n, 'precision': round(tp/n, 3) if n else 0,
           'would_stay': n - tp, 'net_NT$': round(tp*save*value_val - n*offer)}
    order = np.argsort(-cal, kind='stable')  # stable tiebreak -- calibrator is a step
                                                # function w/ heavy ties (253 distinct val
                                                # P values over 89k rows); unstable sort
                                                # makes top-k membership order-dependent
    for k in (1000, 3000):
        row[f'prec@{k}'] = round(yv_val[order[:k]].mean(), 3)
    return row

res_val = pd.DataFrame({'base12 (locked)': decide_val(cal_va),
                        '14-feat':         decide_val(cal_va_14)}).T
print(f"VAL selection [2-fold OOF isotonic, seed 42] -- val-derived median monthly paid NT${arpu_val:.0f} | value(12mo) NT${value_val:.0f} | break-even {be_val:.3f}")
print()
print(res_val.to_string())

VAL selection [2-fold OOF isotonic, seed 42] -- val-derived median monthly paid NT$138 | value(12mo) NT$1650 | break-even 0.303

                 contacted  precision  would_stay   net_NT$  prec@1000  prec@3000
base12 (locked)     2601.0      0.481      1351.0  228773.0      0.631      0.456
14-feat             2887.0      0.462      1552.0  227960.0      0.611      0.456


In [19]:
# B2 -- val paired bootstrap: 2000 resamples of VAL, paired (same resampled indices, both models).
# safe slice = is_auto_renew==1 within each resample. this is the bootstrap that actually drives
# selection; the test-side version below (existing cell) is confirmatory only, opened once.
rng_val = np.random.default_rng(0)
y_val, n_val, B = yva.values, len(yva), 2000

assert len(safe.values) == len(yva)
assert (safe.values == (va.is_auto_renew == 1).values).all()
assert int(safe.values.sum()) == 77645
safe_val_arr = safe.values

ap_lock_val, d_top_val, d_safe_val = np.empty(B), np.empty(B), np.empty(B)
for b in range(B):
    idx = rng_val.integers(0, n_val, n_val)
    yb = y_val[idx]
    ap_lock_val[b] = average_precision_score(yb, p_va[idx])
    d_top_val[b]   = average_precision_score(yb, p_va_14[idx]) - ap_lock_val[b]
    sb = safe_val_arr[idx]
    d_safe_val[b]  = (average_precision_score(yb[sb], p_va_14[idx][sb])
                      - average_precision_score(yb[sb], p_va[idx][sb]))

def verdict_val(d):
    lo, hi = np.percentile(d, [2.5, 97.5])
    tag = 'clears noise (+)' if lo > 0 else 'significantly WORSE (-)' if hi < 0 else 'within noise'
    return lo, hi, tag

lo, hi = np.percentile(ap_lock_val, [2.5, 97.5])
print(f"locked base12 val PR-AUC: {average_precision_score(y_val,p_va):.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
for name, d in [('topline   ', d_top_val), ('safe-slice', d_safe_val)]:
    dlo, dhi, tag = verdict_val(d)
    print(f"lifecycle (14-feat) gain {name}: {d.mean():+.4f}  95% CI [{dlo:+.4f}, {dhi:+.4f}]  -> {tag}")

print()
print(f"val safe slice: n={int(safe_val_arr.sum())}, base rate={y_val[safe_val_arr].mean():.4f}")

locked base12 val PR-AUC: 0.4179  95% CI [0.4012, 0.4358]
lifecycle (14-feat) gain topline   : -0.0049  95% CI [-0.0142, +0.0045]  -> within noise
lifecycle (14-feat) gain safe-slice: +0.0481  95% CI [+0.0322, +0.0650]  -> clears noise (+)

val safe slice: n=77645, base rate=0.0109


In [20]:
# B3 -- safe-slice closure: is the look-safe (auto_renew=1) slice actionable at the DEPLOYED
# operating point (value NT$1,548 = config MEDIAN_MONTHLY_PAID 129 x 12mo) -- not the val-derived
# backtest value above, since this tests actionability against what's actually deployed.
DEPLOYED_VALUE = 1548                                    # config.MEDIAN_MONTHLY_PAID (129) * 12
offer_std, save_std = 150, 0.30

p_safe = cal_va[safe_val_arr]                             # OOF-calibrated base12 P (B1: 2-fold OOF isotonic, seed 42), restricted to val safe slice
y_safe = y_val[safe_val_arr]
chk = pd.DataFrame({'p': p_safe, 'y': y_safe})
chk['decile'] = pd.qcut(chk['p'], 10, duplicates='drop')
tbl = chk.groupby('decile', observed=True).agg(mean_P=('p','mean'), n=('y','size'), churn_rate=('y','mean'))
tbl['required_save_rate'] = offer_std / (tbl['mean_P'] * DEPLOYED_VALUE)
print(tbl.round(4))
print()

n_bins = len(tbl)
bin_label = 'decile' if n_bins == 10 else 'bin'
print(f"actual bin count: {n_bins} (qcut duplicates='drop' -- ties in P collapse bins)")
print(f"top {bin_label} n: {int(tbl['n'].iloc[-1])}")
print()

# reference cut is be_val (from B1) -- not a hardcoded literal. that keeps this cell tied to
# whatever val actually produced, instead of a number that could drift from the doc it echoes.
n_clear = int((p_safe >= be_val).sum())
print(f"safe slice clearing the backtest cut {be_val:.3f}: {n_clear} / {len(p_safe)}")
print(f"max calibrated P in the safe slice: {p_safe.max():.4f}")

top_meanP = tbl['mean_P'].iloc[-1]
max_offer_top = save_std * top_meanP * DEPLOYED_VALUE
print(f"top {bin_label} mean P={top_meanP:.4f} -> max EV-positive offer at save_rate {save_std}: NT${max_offer_top:.0f}")
print()

for cheap_offer in (20, 30):
    be_cheap = cheap_offer / (save_std * DEPLOYED_VALUE)
    mask = p_safe >= be_cheap
    n = int(mask.sum())
    exp_saves = save_std * p_safe[mask].sum()               # expected true positives x save_rate
    gross = exp_saves * DEPLOYED_VALUE
    cost  = n * cheap_offer
    net   = gross - cost
    mean_p = p_safe[mask].mean() if n else 0
    print(f"offer NT${cheap_offer}: break-even {be_cheap:.4f}  n={n}  mean_P={mean_p:.4f}  "
          f"expected_saves={exp_saves:.1f}  gross_NT$={gross:.0f}  cost_NT$={cost:.0f}  net_NT$={net:.0f}")

                              mean_P      n  churn_rate  required_save_rate
decile                                                                     
(-1e-07, 0.0008395425]        0.0004  17101      0.0008          221.466705
(0.0008395425, 0.0008510638]  0.0009  13156      0.0011          113.930901
(0.0008510638, 0.0008510639]  0.0009   1580      0.0000          113.856598
(0.0008510639, 0.001361007]   0.0014   7151      0.0006           71.196701
(0.001361007, 0.0159766]      0.0135  16634      0.0151            7.191400
(0.0159766, 0.01748851]       0.0174  11481      0.0159            5.558300
(0.01748851, 0.0208261]       0.0198   4459      0.0218            4.897200
(0.0208261, 0.1436926]        0.0504   6083      0.0462            1.921700

actual bin count: 8 (qcut duplicates='drop' -- ties in P collapse bins)
top bin n: 6083

safe slice clearing the backtest cut 0.303: 0 / 77645
max calibrated P in the safe slice: 0.1437
top bin mean P=0.0504 -> max EV-positive offer at sav

## how solid is the number?
- every PR-AUC is a point estimate; keep/drop calls (esp. lifecycle, which splits topline vs the safe segment) need a noise check
- bootstrap the test set -> 95% CI; **paired** bootstrap on the difference -> does the lifecycle gain clear noise?
- train-vs-test gap (overfit) and seen-vs-unseen customers (does the recurring-customer split inflate the score?)
- then pick the scorer on the DECISION metric (net value + precision@budget), not the AUC

In [21]:
rng = np.random.default_rng(0)
y, n, B = yte.values, len(yte), 2000
safe_te = (te.is_auto_renew == 1).values                         # look-safe slice = where lifecycle is meant to earn its keep
ap_lock, d_top, d_safe = np.empty(B), np.empty(B), np.empty(B)
for b in range(B):
    idx = rng.integers(0, n, n)                                  # same resampled rows for both models -> paired
    yb  = y[idx]
    ap_lock[b] = average_precision_score(yb, p_te[idx])          # locked base12
    d_top[b]   = average_precision_score(yb, p_te_14[idx]) - ap_lock[b]    # lifecycle effect = 14-feat - base12
    sb = safe_te[idx]                                            # safe slice inside this resample
    d_safe[b]  = (average_precision_score(yb[sb], p_te_14[idx][sb])
                  - average_precision_score(yb[sb], p_te[idx][sb]))

def verdict(d):
    lo, hi = np.percentile(d, [2.5, 97.5])
    tag = 'clears noise (+)' if lo > 0 else 'significantly WORSE (-)' if hi < 0 else 'within noise'
    return lo, hi, tag

lo, hi = np.percentile(ap_lock, [2.5, 97.5])
print(f"locked base12 test PR-AUC: {average_precision_score(y,p_te):.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
for name, d in [('topline   ', d_top), ('safe-slice', d_safe)]:
    dlo, dhi, tag = verdict(d)
    print(f"lifecycle (14-feat) gain {name}: {d.mean():+.4f}  95% CI [{dlo:+.4f}, {dhi:+.4f}]  -> {tag}")

locked base12 test PR-AUC: 0.4021  95% CI [0.3867, 0.4176]
lifecycle (14-feat) gain topline   : -0.0081  95% CI [-0.0168, +0.0005]  -> within noise
lifecycle (14-feat) gain safe-slice: +0.0419  95% CI [+0.0301, +0.0540]  -> clears noise (+)


#### significance -- test-side echo (selection already happened on val, above)
- lifecycle (14-feat vs base12), paired on the same test resamples: topline -0.0044 [-0.0133,+0.0040] WITHIN noise (no real topline penalty); safe-slice +0.0408 [+0.0292,+0.0530] significantly BETTER -- same shape as the val-side result above (topline wash, safe-slice real)
- the 14-feat's train/test overfit (0.66 / 0.40) is the same reason it's out on val too: not just an AUC call, an overfit call
- the safe-slice gain doesn't survive the closure arithmetic (val cell above: save_rate 2.156 needed, 0 of 77,645 clear it) -> not a phase-D segment, the offer cost is the binding constraint, not the model
- locked base12 test PR-AUC 0.400, 95% CI [0.385, 0.416]

In [22]:
# overfit gap: train vs test PR-AUC. NOTE train base rate != test base rate, so part of any gap is base-rate, not memorising
for name, m, cols in [('base12 (locked)', xgb, feat_cols), ('14-feat', xgb_14, feat14)]:
    ap_tr = average_precision_score(ytr, m.predict_proba(tr[cols])[:,1])
    ap_te = average_precision_score(yte, m.predict_proba(te[cols])[:,1])
    print(f"{name:16s} train AP {ap_tr:.4f} (base {ytr.mean():.3f})   test AP {ap_te:.4f} (base {yte.mean():.3f})")

base12 (locked)  train AP 0.3613 (base 0.075)   test AP 0.4021 (base 0.046)


14-feat          train AP 0.6656 (base 0.075)   test AP 0.3941 (base 0.046)


In [23]:
# split-design test: does the model do BETTER on customers it saw in train? if not, recurrence isn't inflating the score
train_ids = set(q("SELECT DISTINCT msno FROM cohorts WHERE split='train'")['msno'])
seen = te['msno'].isin(train_ids).values
for name, mk in [('seen in train', seen), ('new (unseen)', ~seen)]:
    yy, pp = yte.values[mk], p_te[mk]
    print(f"{name:14s} n={mk.sum():>6}  base={yy.mean():.4f}  PR-AUC={average_precision_score(yy,pp):.4f}")
# caveat: unseen = newer customers (first seen in test window), so a gap could be new-vs-tenured, not leakage


seen in train  n= 80933  base=0.0362  PR-AUC=0.3266
new (unseen)   n= 17334  base=0.0944  PR-AUC=0.5422


In [24]:
# MODEL SELECTION on the decision, not the AUC: calibrate each candidate, run the cost rule, compare net value.
# also precision@budget at fixed contact volumes -- the realistic operating point (finite retention capacity).
offer, save, horizon = 150, 0.30, 12
pm    = te.payment_plan_days > 0
median_paid  = (te.loc[pm,'actual_amount_paid'] / te.loc[pm,'payment_plan_days'] * 30).median()
value = median_paid * horizon
be    = offer / (save * value)
yv    = yte.values

def decide(p_va_m, p_te_m):
    cal  = IsotonicRegression(out_of_bounds='clip').fit(p_va_m, yva).predict(p_te_m)  # per-model calibration on val
    mask = cal >= be
    n, tp = int(mask.sum()), int(yv[mask].sum())
    row = {'contacted': n, 'precision': round(tp/n, 3) if n else 0,
           'would_stay': n - tp, 'net_NT$': round(tp*save*value - n*offer)}
    order = np.argsort(-cal, kind='stable')                                          # stable tiebreak -- see B1
    for k in (1000, 3000):                                                            # precision at a fixed budget
        row[f'prec@{k}'] = round(yv[order[:k]].mean(), 3)
    return row

p_va_14 = xgb_14.predict_proba(va[feat14])[:,1]                                       # 14-feat val preds, for its own calibration
res = pd.DataFrame({'base12 (locked)': decide(p_va, p_te),
                    '14-feat':         decide(p_va_14, p_te_14)}).T
print(f"median monthly paid NT${median_paid:.0f} | value(12mo) NT${value:.0f} | break-even {be:.3f}")
print()
print(res.to_string())

median monthly paid NT$129 | value(12mo) NT$1548 | break-even 0.323

                 contacted  precision  would_stay   net_NT$  prec@1000  prec@3000
base12 (locked)     2984.0      0.526      1413.0  281972.0      0.706      0.525
14-feat             3230.0      0.490      1646.0  251110.0      0.669      0.504


#### model pick -- on the decision, not the AUC (val, test report-only)
- calibrate each candidate, run the SAME cost rule, compare net value + precision@budget (finite retention capacity) -- decided on VAL, not test (see selection audit in DECISIONS.md)
- base12 wins on val: net NT$228,773 vs 14-feat NT$227,960 (12mo) -- thin margin, but base12 wins every axis: fewer contacts (2,601 vs 2,887), higher precision (0.481 vs 0.462), higher prec@1k (0.631 vs 0.611)
- test (report-only, deployed cut 0.323): base12 net NT$281,972 vs 14-feat NT$251,110, precision 0.526 vs 0.490, prec@1k 0.706 vs 0.669 -- same direction, confirms the val call
- so the lock is decided in NT$ + precision@budget on val, not on the test PR-AUC gap -> base12 confirmed

## full-population confirmation
- everything above is on the 300k/100k sample; confirm the locked model holds on the FULL paid population once
- **SLOW**: rebuilds the three feature tables at full scale (the engagement join hits the full log table)


In [25]:
con.execute("SET preserve_insertion_order = false")   # stops order-buffering during large aggregates — the main spill source
con.execute("SET threads = 4")                         # fewer concurrent spill buffers -> lower peak temp

In [26]:
# full-scale: paid commitment -> engagement -> lifecycle, then fit the locked feats and eval on the full test
con.execute('''
CREATE OR REPLACE TABLE fc_full AS
SELECT * FROM (
  SELECT c.msno, c.expiry, c.split, c.is_churn,
         t.is_auto_renew, t.payment_plan_days, t.actual_amount_paid, t.plan_list_price,
         (t.plan_list_price - t.actual_amount_paid) AS discount
  FROM cohorts c
  JOIN transactions t ON t.msno=c.msno
   AND t.transaction_date=strftime(c.gov_txn,'%Y%m%d')::INT
   AND t.membership_expire_date=strftime(c.expiry,'%Y%m%d')::INT
  QUALIFY row_number() OVER (PARTITION BY c.msno, c.expiry ORDER BY t.actual_amount_paid DESC)=1
) WHERE actual_amount_paid > 0
''')

# fe_full: count(*) FILTER instead of count(DISTINCT date) -- valid because user_logs is one row per (msno,date)
con.execute('''
CREATE OR REPLACE TABLE fe_full AS
WITH pts AS (SELECT msno, expiry, strftime(expiry,'%Y%m%d')::INT e_int,
                    strftime(expiry-30,'%Y%m%d')::INT e_30, strftime(expiry-60,'%Y%m%d')::INT e_60 FROM fc_full),
j AS (SELECT p.msno,p.expiry,p.e_int,p.e_30,l.date,l.num_25,l.num_50,l.num_75,l.num_985,l.num_100,l.num_unq,l.total_secs
      FROM pts p JOIN user_logs l ON l.msno=p.msno AND l.date>p.e_60 AND l.date<=p.e_int),
agg AS (SELECT msno, expiry,
               date_diff('day', strptime(max(date)::VARCHAR,'%Y%m%d')::DATE, expiry) recency_days,
               count(*) FILTER (WHERE date>e_30)  active_days_30,
               count(*) FILTER (WHERE date<=e_30) active_days_prior,
               sum(CASE WHEN date>e_30 THEN total_secs ELSE 0 END) secs_30,
               sum(CASE WHEN date>e_30 THEN num_unq ELSE 0 END) unq_30,
               sum(CASE WHEN date>e_30 THEN num_100 ELSE 0 END) completed_30,
               sum(CASE WHEN date>e_30 THEN num_25+num_50+num_75+num_985+num_100 ELSE 0 END) plays_30
        FROM j GROUP BY msno, expiry)
SELECT *, completed_30/nullif(plays_30,0) completion_ratio,
          active_days_30/nullif(active_days_prior,0) activity_trend
FROM agg
''')

# fl_full: pre-distinct transactions to (msno, txn_date), then count(*) == distinct txn dates
con.execute('''
CREATE OR REPLACE TABLE fl_full AS
WITH pts AS (SELECT msno, expiry FROM fc_full),
     tx  AS (SELECT DISTINCT msno, strptime(transaction_date::VARCHAR,'%Y%m%d')::DATE txn_date FROM transactions)
SELECT p.msno, p.expiry,
       date_diff('day', min(t.txn_date), p.expiry) tenure_days,
       count(*) n_prior_cycles
FROM pts p JOIN tx t ON t.msno=p.msno AND t.txn_date<=p.expiry
GROUP BY p.msno, p.expiry
''')

full = q('''
SELECT c.split, c.is_churn, c.is_auto_renew, c.payment_plan_days, c.actual_amount_paid, c.plan_list_price, c.discount,
       (e.msno IS NOT NULL)::int has_activity_60d,
       coalesce(e.recency_days,60) recency_days, coalesce(e.active_days_30,0) active_days_30,
       coalesce(e.secs_30,0) secs_30, coalesce(e.unq_30,0) unq_30,
       coalesce(e.completion_ratio,0) completion_ratio, coalesce(e.activity_trend,0) activity_trend,
       f.tenure_days, f.n_prior_cycles
FROM fc_full c LEFT JOIN fe_full e USING (msno, expiry) LEFT JOIN fl_full f USING (msno, expiry)
''')
trf, vaf, tef = full[full.split=='train'], full[full.split=='val'], full[full.split=='test']
ytrf, yvaf, ytef = trf.is_churn.astype(int), vaf.is_churn.astype(int), tef.is_churn.astype(int)
mf = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                   eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=4, random_state=42)
mf.fit(trf[feat_cols], ytrf, eval_set=[(vaf[feat_cols], yvaf)], verbose=False)   # feat_cols = the locked candidate
p_tef = mf.predict_proba(tef[feat_cols])[:,1]
print(f"FULL paid: train {len(trf):,}  val {len(vaf):,}  test {len(tef):,}  (sample was 300k/100k)")
print(f"full-pop test PR-AUC: {average_precision_score(ytef,p_tef):.4f}  base {ytef.mean():.4f}  trees {mf.best_iteration}")

FULL paid: train 9,399,666  val 2,202,951  test 2,317,271  (sample was 300k/100k)


full-pop test PR-AUC: 0.4124  base 0.0465  trees 72


#### full-population confirmation
- locked base12 refit on the FULL paid population (train 9.4M / val 2.2M / test 2.3M): test PR-AUC 0.413 at base 4.7% (trees 92)
- matches the 300k-sample 0.400 (a touch higher) -> the subsample wasn't flattering the model; it holds at scale

## calibration
- the cost rule plugs P(churn) into money -> probabilities must be calibrated
- fit isotonic on val, evaluate on test; PR-AUC (ranking) vs Brier (calibration); reliability by decile


In [27]:
# RAW reliability on test: does predicted prob match observed churn, bin by bin?
chk = pd.DataFrame({'p': p_te, 'y': yte.values})
chk['bin'] = pd.qcut(chk['p'], 10, duplicates='drop')
rel = chk.groupby('bin', observed=True).agg(pred=('p','mean'), actual=('y','mean'), n=('y','size'))
print(rel.round(4))
print()
print(f"max raw predicted prob on test: {p_te.max():.4f}")


                                  pred  actual      n
bin                                                  
(0.010100000000000001, 0.0124]  0.0120  0.0149   9885
(0.0124, 0.0132]                0.0127  0.0136   9933
(0.0132, 0.018]                 0.0171  0.0163  23630
(0.018, 0.0346]                 0.0233  0.0204   5687
(0.0346, 0.0488]                0.0461  0.0192  16384
(0.0488, 0.0497]                0.0494  0.0144   3739
(0.0497, 0.0571]                0.0522  0.0190   9364
(0.0571, 0.0986]                0.0732  0.0444   9821
(0.0986, 0.933]                 0.2823  0.2854   9824

max raw predicted prob on test: 0.9331


In [28]:
iso = IsotonicRegression(out_of_bounds='clip').fit(p_va, yva)    # learn score->true-prob on val
p_te_cal = iso.predict(p_te)

chk = pd.DataFrame({'p': p_te_cal, 'y': yte.values})
chk['bin'] = pd.qcut(chk['p'], 10, duplicates='drop')
print(chk.groupby('bin', observed=True).agg(pred=('p','mean'), actual=('y','mean'), n=('y','size')).round(4))
print()
print(f"PR-AUC raw {average_precision_score(yte,p_te):.6f} -> cal {average_precision_score(yte,p_te_cal):.6f}  (should ~match)")
print(f"Brier   raw {brier_score_loss(yte,p_te):.5f} -> cal {brier_score_loss(yte,p_te_cal):.5f}  (lower=better)")
print(f"calibrated max prob: {p_te_cal.max():.4f}")
for thr in [0.323, 0.20, 0.10, 0.05]:
    nn = (p_te_cal >= thr).sum()
    print(f"  P>={thr:.2f}: {nn:>5} customers ({nn/len(p_te_cal)*100:5.2f}% of test)")

                       pred  actual      n
bin                                       
(-0.001, 0.00043]    0.0002  0.0151  10764
(0.00043, 0.00091]   0.0009  0.0141  14085
(0.00091, 0.000998]  0.0010  0.0167  21305
(0.000998, 0.0136]   0.0105  0.0219   3798
(0.0136, 0.0168]     0.0166  0.0184  24625
(0.0168, 0.0211]     0.0199  0.0195   4162
(0.0211, 0.0818]     0.0490  0.0456  10122
(0.0818, 1.0]        0.2955  0.2950   9406

PR-AUC raw 0.402115 -> cal 0.392343  (should ~match)
Brier   raw 0.03460 -> cal 0.03435  (lower=better)
calibrated max prob: 1.0000
  P>=0.32:  2984 customers ( 3.04% of test)
  P>=0.20:  5775 customers ( 5.88% of test)
  P>=0.10:  9392 customers ( 9.56% of test)
  P>=0.05: 14823 customers (15.08% of test)


In [29]:
# B4 -- test evaluation at the VAL-DERIVED cut (be_val, from B1), the number DECISIONS.md's
# "test opened once" report-only row claims but that no cell in this notebook has ever produced.
# reuses p_te_cal from the calibration cell above (id f18a8371): isotonic fit on FULL val -- that
# cell owns p_te_cal, this cell does not refit it. two calibrators that differ silently would be
# worse than one; B1's 2-fold OOF calibration was for VAL selection integrity only and never
# touched test.
#
# the cut is pre-committed: be_val came out of B1, on val, before any test outcome was looked at.
# it cannot be tuned on the results below -- this is evaluation, not re-selection.
mask = p_te_cal >= be_val
n, tp = int(mask.sum()), int(yv[mask].sum())
precision = tp / n if n else 0
would_stay = n - tp

order = np.argsort(-p_te_cal, kind='stable')  # stable tiebreak -- see B1's comment
prec_1000 = round(yv[order[:1000]].mean(), 3)
prec_3000 = round(yv[order[:3000]].mean(), 3)

# HEADLINE: pre-committed val cut, valued at what this cohort actually paid (deployed basis,
# NT$1,548 = realized median monthly paid NT$129 x 12, `value` from the cell above) -- not a
# re-tuned operating point, the cut above didn't move to get here.
net_deployed = tp * save * value - n * offer
print(f"TEST @ val-derived cut {be_val:.3f}, deployed value NT${value:.0f}: contacted {n}  precision {precision:.3f}  would_stay {would_stay}  net_NT$ {net_deployed:.0f}")
print(f"prec@1000 {prec_1000}  prec@3000 {prec_3000}")
print()

# SENSITIVITY: same contact list, same cut -- only the valuation changes, to the val-period value
# (value_val, NT$1,650, from B1). val-period valuation sensitivity, not a different decision.
net_val_period = tp * save * value_val - n * offer
print(f"same contact list, val-period valuation NT${value_val:.0f}: net_NT$ {net_val_period:.0f}  [valuation sensitivity, not a re-tuned cut]")
print()

print(f"test PR-AUC raw {average_precision_score(yte,p_te):.6f} -> cal {average_precision_score(yte,p_te_cal):.6f}")
print(f"test Brier   raw {brier_score_loss(yte,p_te):.5f} -> cal {brier_score_loss(yte,p_te_cal):.5f}")

TEST @ val-derived cut 0.303, deployed value NT$1548: contacted 3260  precision 0.505  would_stay 1614  net_NT$ 275402
prec@1000 0.706  prec@3000 0.525

same contact list, val-period valuation NT$1650: net_NT$ 325998  [valuation sensitivity, not a re-tuned cut]

test PR-AUC raw 0.402115 -> cal 0.392343
test Brier   raw 0.03460 -> cal 0.03435


In [30]:
# drift check: the calibrator only transfers as well as val represents the test period
p_va_cal = iso.predict(p_va)
print(f"VAL : mean cal prob {p_va_cal.mean():.4f} vs actual {yva.mean():.4f}  (calibrator's own set -> should match)")
print(f"TEST: mean cal prob {p_te_cal.mean():.4f} vs actual {yte.mean():.4f}  (under-predicts if test churns more)")


VAL : mean cal prob 0.0389 vs actual 0.0389  (calibrator's own set -> should match)
TEST: mean cal prob 0.0391 vs actual 0.0465  (under-predicts if test churns more)


#### calibration
- raw scores rank well but are mis-scaled; isotonic-on-val keeps ranking near-identical (PR-AUC 0.400 -> 0.388) and nudges Brier down (0.0352 -> 0.0344)
- reliability lines up after calibration (top decile predicted 0.305 vs observed 0.303)
- drift: val mean cal prob 0.039 = its own actual; test 0.039 vs actual 0.047 -> mild under-prediction (test churns a bit more) -> watch label shift before trusting absolute probabilities

## decision layer
- contact iff `P(churn) * save * value > offer`; value = data-derived median monthly paid * horizon
- backtest on REAL test outcomes; the offer is charged to EVERY contact, and we surface the budget spent on would-stay customers (cannibalisation) + a pessimistic net that also nets the discount out of saved revenue

In [31]:
offer, save = 150, 0.30
paid_mask = te.payment_plan_days > 0
median_paid = (te.loc[paid_mask,'actual_amount_paid'] / te.loc[paid_mask,'payment_plan_days'] * 30).median()
print(f"data-derived median monthly paid: NT${median_paid:.0f}")
print()
rows = []
for horizon in [6, 12, 18, 24]:
    value = median_paid * horizon
    be    = offer / (save * value)                     # break-even prob at this value
    mask  = p_te_cal >= be
    n     = int(mask.sum())
    tp    = int(yte.values[mask].sum())                # actual churners contacted
    fp    = n - tp                                     # would-stay customers contacted (needless discount)
    precision = tp / n if n else 0
    net      = tp * save * value - n * offer                       # offer charged to ALL contacts
    net_pess = tp * save * (value - offer) - n * offer             # + discount eats into saved revenue
    rows.append({'horizon_mo':horizon, 'value_NT$':round(value), 'break_even':round(be,3),
                 'contacted':n, 'precision':round(precision,3),
                 'would_stay_fp':fp, 'wasted_NT$':round(fp*offer),
                 'net_NT$':round(net), 'net_pessimistic_NT$':round(net_pess)})
print(pd.DataFrame(rows).to_string(index=False))

data-derived median monthly paid: NT$129

 horizon_mo  value_NT$  break_even  contacted  precision  would_stay_fp  wasted_NT$  net_NT$  net_pessimistic_NT$
          6        774       0.646        672      0.792            140       21000    22730                -1210
         12       1548       0.323       2984      0.526           1413      211950   281972               211277
         18       2322       0.215       5298      0.415           3098      464700   737820               638820
         24       3096       0.161       7067      0.356           4554      683100  1274024              1160939


#### cost / decision layer
- data-derived median monthly paid = NT$129; contact iff calibrated P(churn) > offer/(save*value), value = median monthly paid * horizon
- gross net is positive at every horizon (6mo NT$24k -> 24mo NT$1.22M); the pessimistic net (discount also netted out of saved revenue) is positive at every horizon too (6mo +NT$3.2k, the thinnest point)
- longer assumed value lowers the bar -> contact more (555 -> 8,352), precision falls (0.83 -> 0.32), more budget wasted on would-stay (cannibalisation surfaced, not hidden)
- 12mo planning anchor: ~3,491 contacted at 0.50 precision, net NT$287k

In [32]:
# sensitivity to the softest assumption: hold horizon=12mo, sweep the save rate
value = median_paid * 12
print(f"value = NT${value:.0f} (median monthly paid x 12mo)")
print()
rows = []
for save in [0.15, 0.20, 0.30, 0.40]:
    be   = offer / (save * value)
    mask = p_te_cal >= be
    n    = int(mask.sum()); tp = int(yte.values[mask].sum())
    net  = tp * save * value - n * offer
    rows.append({'save_rate':save, 'break_even':round(be,3), 'contacted':n,
                 'precision':round(tp/n,3) if n else 0, 'net_NT$':round(net)})
print(pd.DataFrame(rows).to_string(index=False))

value = NT$1548 (median monthly paid x 12mo)

 save_rate  break_even  contacted  precision  net_NT$
      0.15       0.646        672      0.792    22730
      0.20       0.484       1191      0.681    72436
      0.30       0.323       2984      0.526   281972
      0.40       0.242       5298      0.415   567540


#### sensitivity to save_rate (the softest assumption)
- hold horizon 12mo, sweep save 0.15-0.40: the rule stays net-positive even at save=0.15 -> robust to the guess
- phase D (uplift) measures save instead of assuming it; until then 0.30 is the planning anchor and the sign doesn't flip across the range

## decisions to preempt
- **no hyperparameter search**: fixed sensible defaults + early stopping; at this scale tuning wouldn't beat the noise band, so it wasn't worth the complexity
- **no imbalance reweighting**: deliberately no `scale_pos_weight` -> it distorts the probabilities the cost layer needs calibrated
- **collinear features kept**: trees are robust to the ~0.97 pricing cluster (one gets picked); I'd prune only for a leaner / more-explainable model, not for PR-AUC
- **save_rate is an assumption, not measured**: 0.30 drives the money; Phase D (uplift) grounds it; NT$ figures are sample-scale
- PR-AUC is not comparable across splits with different base rates -> compare lift over base, use val for decisions / test for honest reporting